# SPHEREx cutout CNN
Attach the extracted PlanetX v1.2.0 dataset or its original ZIP, enable a GPU, and run all cells. All project code and the previous benchmark are embedded. Save the executed version to retain its files. Download results.json to share metrics, or run.zip to share the metrics and trained checkpoint. The final cell prints the same metrics. Full training uses eight epochs and selects the checkpoint by validation average precision.

In [ ]:
from pathlib import Path
import zipfile
import os
working = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
inputs = Path(os.environ.get("PLANETX_INPUT", "/kaggle/input"))
def find_root(folder):
    return next((p.parent for p in folder.rglob("index.parquet")
                 if (p.parent / "spherex/injected_truth.parquet").is_file()), None)
root = find_root(inputs)
if root is None:
    archives = list(inputs.rglob("planetx-dataset-v1.2.0.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Attach the extracted dataset or one original planetx-dataset-v1.2.0.zip.")
    destination = working / "planetx-input"
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(destination)
    root = find_root(destination)
if root is None:
    raise FileNotFoundError("Missing dataset index or injected truth table.")
print("Dataset:", root)

## Candidate generation and labels
These functions reproduce the original candidates, field splits and two-pixel matching rule.

In [ ]:
"""Blind peak proposals and supervised SPHEREx change classification."""
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve, average_precision_score

FEATURES = ["peak", "flux_z", "reference_z", "diff_z", "dip", "rise",
            "centroid_shift", "flux_contrast", "valid_fraction", "wave_gap"]
CLASSES = ["background", "motion", "variable", "transient"]
KEY = ["realization_id", "frame_index", "obj_id"]


def proposals(path, frames, injected, morphology=False):
    """Return peaks in calibrated, aligned images without access to object labels."""
    with np.load(path) as stack:
        ids = frames.index.to_numpy()
        if injected and not np.array_equal(ids, stack["frame_index"]):
            raise ValueError("Frame metadata does not match image order")
        image, valid = stack["image"], stack["valid"].astype(bool)
        if not injected:
            image, valid = image[ids], valid[ids]
    image = np.where(valid, image, np.nan)
    times = frames.mjd_mid.to_numpy()
    wave = frames.wave_center_um.to_numpy()
    detectors = frames.detector.to_numpy()
    yy, xx = np.mgrid[-2:3, -2:3]
    result = []
    for i, frame_id in enumerate(ids):
        refs = np.flatnonzero((detectors == detectors[i]) & (abs(times - times[i]) > 1))
        if len(refs) < 2:
            continue
        refs = refs[np.argsort(abs(np.log(wave[refs] / wave[i])))[:3]]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ref = np.nanmedian(image[refs], axis=0)
        good = np.isfinite(image[i]) & np.isfinite(ref)
        current = np.where(good, image[i], 0)
        reference = np.where(good, ref, 0)
        diff = gaussian_filter(current - reference, 0.85)
        noise = max(1e-8, 1.4826 * np.median(abs(diff[good] - np.median(diff[good])))) if good.any() else 1
        z = diff / noise
        strength = abs(z)
        peaks = good & (strength >= 3.5) & (strength == maximum_filter(strength, 5))
        peaks[:2] = peaks[-2:] = False
        peaks[:, :2] = peaks[:, -2:] = False
        y, x = np.where(peaks)
        if not len(x):
            continue
        region = (y[:, None, None] + yy, x[:, None, None] + xx)
        a, b, d = current[region], reference[region], z[region]
        def centroid(patch):
            weights = np.maximum(patch - patch.min((1, 2), keepdims=True), 0)
            total = np.maximum(weights.sum((1, 2)), 1e-8)
            return np.array([(weights * axis).sum((1, 2)) / total for axis in (xx, yy)])
        signal = max(float(frames.pixel_noise.iloc[i]), 1e-8)
        flux_a, flux_b = a.sum((1, 2)), b.sum((1, 2))
        features = [strength[y, x], flux_a / signal, flux_b / signal, z[y, x],
                    d.min((1, 2)), d.max((1, 2)), np.linalg.norm(centroid(a)-centroid(b), axis=0),
                    (flux_a-flux_b)/(abs(flux_a)+abs(flux_b)+signal), good[region].mean((1, 2)),
                    np.full(len(x), np.max(abs(np.log(wave[refs]/wave[i]))))]
        table = pd.DataFrame(np.column_stack(features), columns=FEATURES)
        if morphology:
            # Characterize signed structure around the existing peak, without
            # allowing masked pixels or off-image padding to create a lobe.
            dy, dx = np.mgrid[-4:5, -4:5]
            iy, ix = y[:, None, None] + dy, x[:, None, None] + dx
            inside = (iy >= 0) & (iy < z.shape[0]) & (ix >= 0) & (ix < z.shape[1])
            region9 = (iy.clip(0, z.shape[0]-1), ix.clip(0, z.shape[1]-1))
            patch = np.where(inside & good[region9], z[region9], 0)
            signed = patch * np.sign(z[y, x])[:, None, None]
            same, opposite = np.maximum(signed, 0), np.maximum(-signed, 0)
            mass = same.sum((1, 2))
            anti = opposite.sum((1, 2))
            table["opposite_fraction"] = anti / (mass + anti + 1e-8)
            table["opposite_peak_ratio"] = opposite.max((1, 2)) / strength[y, x]
            table["core_fraction"] = same[:, 3:6, 3:6].sum((1, 2)) / mass
            table["lobe_offset"] = np.hypot(*[(opposite * v).sum((1, 2)) /
                                                (anti + 1e-8) for v in (dx, dy)])
            table["spatial_extent"] = (same * (dx*dx+dy*dy)).sum((1, 2)) / mass
        table["frame_index"], table["x"], table["y"] = frame_id, x, y
        result.append(table)
    return pd.concat(result, ignore_index=True) if result else pd.DataFrame(columns=FEATURES+["frame_index","x","y"])


def prepare(root):
    ix = pd.read_parquet(root/"index.parquet").query("mission == 'spherex'")
    frames = pd.read_parquet(root/"spherex/frames.parquet").query("qc_pass")
    objects = pd.read_parquet(root/"spherex/injected_objects.parquet")
    truth = pd.read_parquet(root/"spherex/injected_truth.parquet").merge(
        objects[["realization_id","obj_id","class","is_mover"]], on=["realization_id","obj_id"], validate="many_to_one")
    truth["label"] = np.select([truth["class"].eq("variable"), truth["class"].eq("transient"), truth.is_mover],
                               ["variable","transient","motion"], default="background")
    eligible = truth.query("present and in_grid and masked_frac < 0.5 and not saturated").copy()
    return ix, frames, eligible


def collect(root, ix, frames, eligible, enrich=None, morphology=False):
    samples = []
    for row in ix.query("sample_type == 'injected'").itertuples():
        meta = frames[frames.stack_id == row.stack_id].set_index("frame_index").sort_index()
        candidates = proposals(root/row.path, meta, True, morphology=morphology)
        if enrich is not None:
            candidates = enrich(root/row.path, meta, candidates, injected=True)
        candidates["obj_id"], candidates["label"] = -1, "background"
        labels = eligible[eligible.realization_id == row.realization_id]
        for fid, group in candidates.groupby("frame_index"):
            stars = labels[labels.frame_index == fid]
            if stars.empty:
                continue
            dist, idx = cKDTree(stars[["x","y"]]).query(group[["x","y"]])
            near = dist <= 2
            candidates.loc[group.index[near], ["obj_id","label"]] = stars.iloc[idx[near]][["obj_id","label"]].to_numpy()
        candidates["realization_id"], candidates["fold"] = row.realization_id, row.fold
        samples.append(candidates)
    return pd.concat(samples, ignore_index=True)

## Image model
Each 11 by 11 cutout has four channels: the current exposure, reference, difference and validity mask. All validation and test proposals are retained. The training loss weights classes by their inverse square-root frequency.

In [ ]:
"""Train a cutout classifier and export the run for review."""
import copy
import json
import warnings

import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_recall_fscore_support


def patches(path, meta, peaks, injected):
    with np.load(path) as archive:
        image = np.where(archive["valid"], archive["image"], np.nan)
        assert np.array_equal(archive["frame_index"], meta.index)
    cuts = np.empty((len(peaks), 4, 11, 11), dtype=np.float16)
    dy, dx = np.mgrid[-5:6, -5:6]
    for fid, group in peaks.groupby("frame_index"):
        i = meta.index.get_loc(fid)
        row = meta.iloc[i]
        refs = np.flatnonzero((meta.detector == row.detector) & (abs(meta.mjd_mid-row.mjd_mid)>1))
        refs = refs[np.argsort(abs(np.log(meta.wave_center_um.to_numpy()[refs]/row.wave_center_um)))[:3]]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ref = np.nanmedian(image[refs], axis=0)
        y = group.y.to_numpy()[:,None,None] + dy
        x = group.x.to_numpy()[:,None,None] + dx
        inside = (y>=0)&(x>=0)&(y<ref.shape[0])&(x<ref.shape[1])
        region = y.clip(0,ref.shape[0]-1), x.clip(0,ref.shape[1]-1)
        a, b = image[i][region], ref[region]
        valid = inside & np.isfinite(a) & np.isfinite(b)
        noise = float(row.pixel_noise)
        if not np.isfinite(noise) or noise <= 0:
            pixels = image[i][np.isfinite(image[i])].astype(np.float64)
            noise = 1.4826 * np.median(np.abs(pixels - np.median(pixels)))
            if noise <= 0:
                noise = pixels.std()
        if not np.isfinite(noise):
            raise ValueError(f"Cannot estimate pixel noise in {path}, frame {fid}")
        noise = max(noise, 1e-8)
        channels = [np.arcsinh(np.where(valid,v,0)/noise).clip(-16,16) for v in (a,b,a-b)]
        cuts[group.index] = np.stack([*channels,valid],axis=1)
    if not np.isfinite(cuts).all():
        raise ValueError(f"Non-finite image cutouts in {path}")
    return cuts


def train_cutouts(root, out, reference, smoke=False):
    torch.manual_seed(42)
    rng = np.random.default_rng(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out.mkdir(parents=True,exist_ok=True)
    ix, frames, truth = prepare(root)
    ix = ix.query("sample_type == 'injected'").copy()
    assert ix.groupby("group").fold.nunique().max()==1
    if smoke:
        ix = ix.groupby("fold",sort=False).head(1)
    cut_blocks, labels = [], []
    for j, row in enumerate(ix.itertuples()):
        meta = frames[frames.stack_id==row.stack_id].set_index("frame_index").sort_index()
        block = collect(root, ix.iloc[j:j+1], frames, truth)
        if row.fold >= 2:
            bg = block.index[block.label=="background"]
            take = rng.choice(bg,min(len(bg),max(128,8*(len(block)-len(bg)))),replace=False)
            block = block.loc[np.sort(np.r_[block.index[block.label!="background"],take])]
        block = block.reset_index(drop=True)
        cut_blocks.append(patches(root/row.path,meta,block,True))
        labels.append(block[["fold","label","realization_id","obj_id"]])
        if j % 20 == 0:
            print("Prepared stacks", j+1, "/",len(ix),flush=True)
    table = pd.concat(labels,ignore_index=True)
    X = torch.from_numpy(np.concatenate(cut_blocks))
    del cut_blocks
    y = torch.tensor(pd.Categorical(table.label,categories=CLASSES).codes.astype("int64"))
    def loader(mask,shuffle=False):
        idx = torch.from_numpy(np.flatnonzero(mask))
        return DataLoader(idx,batch_size=256,shuffle=shuffle)
    model = nn.Sequential(nn.Conv2d(4,24,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.Flatten(),
                          nn.Linear(48*5*5,64),nn.ReLU(),nn.Linear(64,len(CLASSES))).to(device)
    train = table.fold>=2
    weights = torch.bincount(y[train.to_numpy()],minlength=4).clamp(min=1).float().rsqrt()
    loss_fn = nn.CrossEntropyLoss(weight=weights.to(device))
    opt = torch.optim.Adam(model.parameters(),lr=.001)
    def predict(mask):
        model.eval()
        with torch.no_grad():
            prob = np.concatenate([model(X[idx].to(device,dtype=torch.float32)).softmax(1).cpu().numpy() for idx in loader(mask)])
            if not np.isfinite(prob).all():
                raise FloatingPointError("Non-finite model predictions before evaluation")
            return prob
    validation = table.fold==1
    best = -1.
    history = []
    for epoch in range(1, (1 if smoke else 8)+1):
        model.train()
        for idx in loader(train,True):
            opt.zero_grad()
            loss = loss_fn(model(X[idx].to(device,dtype=torch.float32)),y[idx].to(device))
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss in epoch {epoch}")
            loss.backward()
            opt.step()
        scores = 1-predict(validation)[:,0]
        actual = table.loc[validation,"label"]!="background"
        ap = float(average_precision_score(actual,scores))
        history.append(ap)
        if ap>best:
            best, state, selected = ap, copy.deepcopy(model.state_dict()), epoch
            p,r,t = precision_recall_curve(actual,scores)
            cut = float(t[np.argmax((2*p*r/np.maximum(p+r,1e-10))[:-1])])
        print("Epoch",epoch,"validation average precision",round(ap,4),flush=True)
    model.load_state_dict(state)
    test = table.fold==0
    data = table[test].reset_index(drop=True)
    prob = predict(test.to_numpy())
    score = 1-prob[:,0]
    kinds = np.array(CLASSES)[1+prob[:,1:].argmax(axis=1)]
    actual = data.label.ne("background")
    eligible = truth[truth.realization_id.isin(ix.loc[ix.fold==0,"realization_id"])]
    totals = eligible.drop_duplicates(["realization_id","obj_id"]).label.value_counts()
    def metrics(flag):
        p,r,f,_ = precision_recall_fscore_support(actual,flag,average="binary",zero_division=0)
        recovered = data[flag & data.label.eq(kinds)].drop_duplicates(["realization_id","obj_id"]).label.value_counts()
        return dict(alerts=int(flag.sum()),precision=float(p),recall=float(r),f1=float(f),
                    object_hits={c:int(recovered.get(c,0)) for c in CLASSES[1:]},
                    transient_alerts=int((flag & (kinds=="transient")).sum()),
                    transient_correct=int((flag & (kinds=="transient") & data.label.eq("transient")).sum()))
    budget = reference["alerts"]
    if smoke:
        budget = max(1, round(len(data)*4453/117825))
    else:
        assert len(data)==117825 and len(ix)==198, "Dataset does not match the published benchmark"
    top = np.zeros(len(data),dtype=bool)
    top[np.argsort(-score,kind="stable")[:budget]] = True
    matched = metrics(top)
    result = dict(status="complete",smoke=smoke,device=str(device),torch_version=str(torch.__version__),
                  seed=42,selected_epoch=selected,validation_ap=history,threshold=cut,
                  test_candidates=len(data),test_average_precision=float(average_precision_score(actual,score)),
                  eligible_objects={c:int(totals.get(c,0)) for c in CLASSES[1:]},
                  equal_budget=matched,operating_point=metrics(score>=cut))
    if not smoke:
        result.update(reference_equal_budget=reference,f1_change_points=100*(matched["f1"]-reference["f1"]))
    torch.save(dict(state_dict={k:v.cpu() for k,v in state.items()},classes=CLASSES,threshold=cut),out/"model.pt")
    (out/"results.json").write_text(json.dumps(result,indent=2))
    with zipfile.ZipFile(out/"run.zip","w",zipfile.ZIP_DEFLATED) as archive:
        for name in ["model.pt","results.json"]:
            archive.write(out/name,name)
    return result

In [ ]:
reference = {'alerts': 4453, 'precision': 0.682685829777678, 'recall': 0.6141414141414141, 'f1': 0.6466021482505583, 'average_precision': 0.699734487031953, 'object_hits': {'motion': 117, 'variable': 46}, 'transient_alerts': 0, 'transient_correct': 0}
result = train_cutouts(root, working / 'spherex-cnn', reference)
print(json.dumps(result, indent=2))
print('Share:', working / 'spherex-cnn/results.json')